# Internal negative data

In [3]:
import pandas as pd
import os

data_dir = "../data"

all = pd.read_csv(os.path.join(data_dir, "processed", "all_molecules.csv"))
all = all[all["activity"]==0]
all_ = all[["smiles", "activity"]]

all_.to_csv(os.path.join(data_dir, "processed","neg_samplers", "internal_all_negs.csv"), index=False)

sd = all[all["category"]=="synthetic"]
sd = sd[["smiles", "activity"]]
sd.to_csv(os.path.join(data_dir, "processed","neg_samplers", "internal_sd_negs.csv"), index=False)

np = all[all["category"]=="natural"]
np = np[["smiles", "activity"]]
np.to_csv(os.path.join(data_dir, "processed", "neg_samplers", "internal_np_negs.csv"), index=False)

target = pd.read_csv(os.path.join(data_dir, "processed", "ACE.csv"))
target = target[target["activity"]==0]
target = target[["smiles", "activity"]]
target.to_csv(os.path.join(data_dir, "processed","neg_samplers", "internal_ACE_negs.csv"), index=False)

target = pd.read_csv(os.path.join(data_dir, "processed", "CCBs.csv"))
target = target[target["activity"]==0]
target = target[["smiles", "activity"]]
target.to_csv(os.path.join(data_dir, "processed","neg_samplers", "internal_CCBs_negs.csv"),index=False)

# Negative Decoys with eos3e6s

In [ ]:
import pandas as pd
import os

data_dir = "../data"

all = pd.read_csv(os.path.join(data_dir, "processed", "all_molecules.csv"))
all = all[all["activity"]==1]
smi = all["smiles"].tolist()
print(len(smi))
all = all[["smiles"]]
all.to_csv(os.path.join(data_dir, "processed", "neg_samplers", "positive_smiles.csv"), index=False)

749


In [16]:
targets = ["ACE", "CCBs", "AT1RBs"]
for t in targets:
    df = pd.read_csv(os.path.join(data_dir, "processed", f"{t}.csv"))
    df = df[df["activity"]==1]
    smi = df["smiles"].tolist()
    print(len(smi))
    df = df[["smiles"]]
    df.to_csv(os.path.join(data_dir, "processed", "neg_samplers", f"{t}_positive_smiles.csv"), index=False)

245
159
153


# Evaluate possible datasets

### General data

In [11]:
all = pd.read_csv(os.path.join(data_dir, "processed", "all_molecules.csv"))
print("Total mols:", len(all), ", Total positives:", len(all[all["activity"]==1]), 
      ", Negative mols:",len(all[all["activity"]==0]), 
      ", Positive proportion:", len(all[all["activity"]==1])/len(all))
np = all[all["category"]=="natural"]
print("NP mols:", len(np), ", Total positives:", len(np[np["activity"]==1]), 
      ", Negative mols:",len(np[np["activity"]==0]), 
      ", Positive proportion:", len(np[np["activity"]==1])/len(np))
sd = all[all["category"]=="synthetic"]
print("SD mols:", len(sd), ", Total positives:", len(sd[sd["activity"]==1]), 
      ", Negative mols:",len(sd[sd["activity"]==0]), 
      ", Positive proportion:", len(sd[sd["activity"]==1])/len(sd))

Total mols: 830 , Total positives: 749 , Negative mols: 81 , Positive proportion: 0.9024096385542169
NP mols: 370 , Total positives: 339 , Negative mols: 31 , Positive proportion: 0.9162162162162162
SD mols: 460 , Total positives: 410 , Negative mols: 50 , Positive proportion: 0.8913043478260869


In [ ]:
# Get three internal datasets (total, Np and sd)
all = pd.read_csv(os.path.join(data_dir, "processed", "all_molecules.csv"))
all_ = all[["smiles", "activity"]]
all_.to_csv(os.path.join(data_dir, "processed", "ml_datasets", "allmols_internal.csv"), index=False)
np_ = np[["smiles", "activity"]]
np_.to_csv(os.path.join(data_dir, "processed", "ml_datasets", "np_internal.csv"), index=False)
sd_ = sd[["smiles", "activity"]]
sd_.to_csv(os.path.join(data_dir, "processed", "ml_datasets", "sd_internal.csv"), index=False)

In [13]:
# Get one with all molecules (positive and negative) + external negatives
all = pd.read_csv(os.path.join(data_dir, "processed", "all_molecules.csv"))
all_ = all[["smiles", "activity"]]
neg = pd.read_csv(os.path.join(data_dir, "processed", "neg_samplers", "eos3e6s.csv"))
decoy_columns = [c for c in neg.columns if c.startswith("smi")][:10] # get only 10 negatives per active input
all_decoys = (
    neg[decoy_columns]
    .values
    .flatten()
)
all_decoys = [s for s in all_decoys if isinstance(s, str) and s.strip() != ""]
print("Decoys:",len(all_decoys), len(set(all_decoys)))
inact = pd.DataFrame({"smiles":all_decoys})
inact["activity"]=0
df = pd.concat([all_, inact])
print("Total mols:", len(df), ", Total positives:", len(df[df["activity"]==1]), 
      ", Negative mols:",len(df[df["activity"]==0]), 
      ", Positive proportion:", len(df[df["activity"]==1])/len(df))
df.to_csv(os.path.join(data_dir,"processed", "ml_datasets", f"allmols_eos3e6snegs.csv"), index=False)

Decoys: 7490 7490
Total mols: 8320 , Total positives: 749 , Negative mols: 7571 , Positive proportion: 0.09002403846153846


In [18]:
# SD and NP separated with external decoys
all = pd.read_csv(os.path.join(data_dir, "processed", "all_molecules.csv"))

np = all[all["category"]=="natural"]
np = np[["smiles", "activity"]]
neg = pd.read_csv(os.path.join(data_dir, "processed", "neg_samplers", "eos3e6s.csv"))
neg["smiles"] = all["smiles"]
neg_np = neg[neg["smiles"].isin(np["smiles"].tolist())]
decoy_columns = [c for c in neg_np.columns if c.startswith("smi")][:10] # get only 10 negatives per active input
all_decoys = (
    neg_np[decoy_columns]
    .values
    .flatten()
)
all_decoys = [s for s in all_decoys if isinstance(s, str) and s.strip() != ""]
print("Decoys:",len(all_decoys), len(set(all_decoys)))
inact = pd.DataFrame({"smiles":all_decoys})
inact["activity"]=0
df = pd.concat([np, inact])
print("Total mols:", len(df), ", Total positives:", len(df[df["activity"]==1]), 
      ", Negative mols:",len(df[df["activity"]==0]), 
      ", Positive proportion:", len(df[df["activity"]==1])/len(df))
df.to_csv(os.path.join(data_dir,"processed", "ml_datasets", f"np_eos3e6snegs.csv"), index=False)

sd = all[all["category"]=="synthetic"]
sd = sd[["smiles", "activity"]]
neg = pd.read_csv(os.path.join(data_dir, "processed", "neg_samplers", "eos3e6s.csv"))
neg["smiles"] = all["smiles"]
neg_sd = neg[neg["smiles"].isin(sd["smiles"].tolist())]
decoy_columns = [c for c in neg_sd.columns if c.startswith("smi")][:10] # get only 10 negatives per active input
all_decoys = (
    neg_sd[decoy_columns]
    .values
    .flatten()
)
all_decoys = [s for s in all_decoys if isinstance(s, str) and s.strip() != ""]
print("Decoys:",len(all_decoys), len(set(all_decoys)))
inact = pd.DataFrame({"smiles":all_decoys})
inact["activity"]=0
df = pd.concat([sd, inact])
print("Total mols:", len(df), ", Total positives:", len(df[df["activity"]==1]), 
      ", Negative mols:",len(df[df["activity"]==0]), 
      ", Positive proportion:", len(df[df["activity"]==1])/len(df))
df.to_csv(os.path.join(data_dir,"processed", "ml_datasets", f"sd_eos3e6snegs.csv"), index=False)

Decoys: 3700 3700
Total mols: 4070 , Total positives: 339 , Negative mols: 3731 , Positive proportion: 0.0832923832923833
Decoys: 3790 3790
Total mols: 4250 , Total positives: 410 , Negative mols: 3840 , Positive proportion: 0.09647058823529411


## Target data

In [14]:
targets = ["ACE", "CCBs", "AT1RBs"]

for t in targets:
    df = pd.read_csv(os.path.join(data_dir, "processed", f"{t}.csv"))
    print("Total mols:", len(df), ", Total positives:", len(df[df["activity"]==1]), 
      ", Negative mols:",len(df[df["activity"]==0]), 
      ", Positive proportion:", len(df[df["activity"]==1])/len(df))


Total mols: 272 , Total positives: 245 , Negative mols: 27 , Positive proportion: 0.9007352941176471
Total mols: 164 , Total positives: 159 , Negative mols: 5 , Positive proportion: 0.9695121951219512
Total mols: 153 , Total positives: 153 , Negative mols: 0 , Positive proportion: 1.0


In [15]:
# ACE internal
df = pd.read_csv(os.path.join(data_dir, "processed", "ACE.csv"))
df = df[["smiles", "activity"]]
df.to_csv(os.path.join(data_dir, "processed", "ml_datasets", "ACE_internal.csv"), index=False)

In [20]:
# all with external data
targets = ["ACE", "CCBs", "AT1RBs"]

for t in targets:
    df = pd.read_csv(os.path.join(data_dir, "processed", f"{t}.csv"))
    neg = pd.read_csv(os.path.join(data_dir, "processed","neg_samplers", f"eos3e6s_{t}.csv"))
    decoy_columns = [c for c in neg.columns if c.startswith("smi")][:10] # get only 10 negatives per active input
    all_decoys = (
        neg[decoy_columns]
        .values
        .flatten()
    )
    all_decoys = [s for s in all_decoys if isinstance(s, str) and s.strip() != ""]
    print("Decoys:",len(all_decoys), len(set(all_decoys)))
    inact = pd.DataFrame({"smiles":all_decoys})
    inact["activity"]=0
    df = pd.concat([df, inact])
    print("Total mols:", len(df), ", Total positives:", len(df[df["activity"]==1]), 
        ", Negative mols:",len(df[df["activity"]==0]), 
        ", Positive proportion:", len(df[df["activity"]==1])/len(df))
    df.to_csv(os.path.join(data_dir,"processed", "ml_datasets", f"{t}_eos3e6snegs.csv"), index=False)

Decoys: 2450 2450
Total mols: 2722 , Total positives: 245 , Negative mols: 2477 , Positive proportion: 0.09000734753857458
Decoys: 1590 1590
Total mols: 1754 , Total positives: 159 , Negative mols: 1595 , Positive proportion: 0.09064994298745724
Decoys: 1530 1530
Total mols: 1683 , Total positives: 153 , Negative mols: 1530 , Positive proportion: 0.09090909090909091
